<a href="https://colab.research.google.com/github/pauljit/Military_service_coding/blob/main/Graphic_Engine/Weekly_Study/Turtle_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 8일차(26.02.20) (4x4 행렬 클래스 구현, 스케일 조절, 벡터 적용, 이동, 로테이션)
# 9일차(26.02.23) (행렬 곱, 짐벌락 체험)
# 10일차(26.02.24) (View 행렬, Projection 행렬)
# 11일차(26.02.25) (MVP 융합)
# 12일차(26.02.26) (래스터화, 폴리곤 색칠, 파일 분리 및 정리)


%%writefile mymath.py

import math

class Vector2:
  def __init__(self, x = 0, y = 0):
    self.x = x
    self.y = y

  #벡터끼리 더하기
  def add(self, other_vector):
    return Vector2(self.x + other_vector.x, self.y + other_vector.y)

  #벡터끼리 빼기
  def minus(self, other_vector):
    return Vector2(self.x - other_vector.x, self.y - other_vector.y)

  #벡터의 곱셈
  def multiply(self, factor):
    return Vector2(self.x *factor, self.y * factor)

  #벡터의 길이
  def magnitude(self):
    return math.sqrt((self.x ** 2) + (self.y ** 2))

  #벡터 정보 출력
  def status(self):
    print(f"x: {self.x}, y: {self.y}")

  #벡터 정규화 (방향을 알기 위한 용도)
  def normalize(self):
    mag = self.magnitude()
    if mag == 0:
      return Vector2(0,0)
    else:
      return Vector2(self.x/mag, self.y/mag)

  #벡터 내적 (+-0의 상태에 따라 방향의 일치성 확인, 양에 따라 빛의 반사율 확인)
  def dot(self, other_vector):
    return ((self.x*other_vector.x) + (self.y * other_vector.y))

  #벡터 외적
  def cross(self, other_vector):
    return (self.x * other_vector.y) - (self.y * other_vector.x)



class Vector3:
  # 1. 생성자 (z축 추가)
  def __init__(self, x=0, y=0, z=0):
    self.x = x
    self.y = y
    self.z = z

  # 2. 덧셈 (z축 끼리도 더해주세요)
  def add(self, other):
    return Vector3(self.x + other.x, self.y + other.y, self.z + other.z)

  # 3. 뺄셈
  def minus(self, other):
    return Vector3(self.x - other.x, self.y - other.y, self.z - other.z)

  # 4. 스칼라 곱셈
  def multiply(self, scalar):
    return Vector3(self.x * scalar, self.y * scalar, self.z * scalar)

  # 5. 길이 구하기
  def magnitude(self):
    return math.sqrt(self.x ** 2 + self.y ** 2 + self.z **2)

  # 6. 정규화 (방향 벡터로 만들기)
  def normalize(self):
    mag = self.magnitude()
    if mag == 0:
      return Vector3(0, 0, 0)
    else:
      return Vector3(self.x / mag, self.y / mag, self.z / mag)

  # 7. 내적 (조명 계산의 핵심)(인식 범위)
  def dot(self, other):
    return (self.x * other.x) + (self.y * other.y) + (self.z * other.z)

  # 8. 벡터 좌표 출력 (:.2f는 소수점 2자리까지만 출력한다는 뜻)
  def status(self):
    print(f"Vector3(x: {self.x:.2f}, y: {self.y:.2f}, z: {self.z:.2f})")

  #  9. 위치 벡터끼리의 거리 (서로 뺀 값의 길이)
  def distance(self, other):
    return self.minus(other).magnitude()

  #  10. 벡터끼리의 각도 (서로 정규화한 벡터의 내적의 아크코사인)(적이 인식 각도 확인 가능)
  def angle(self, other):
     dot_product = self.normalize().dot(other.normalize())
     # 1.0001등의 값을 내어 acos가 오류를 내지 않기 위해 -1.0 ~ 1.0으로 바꾸기
     dot_product = max(-1.0, min(1.0, dot_product))
     radian = math.acos(dot_product)
     return math.degrees(radian)

  # 11. 외적 (두 벡터의 수직인 법선 벡터)(삼각 폴리곤의 수직을 구하여 빛 반사 및 culling 최적화)
  def cross(self, other):
    normal_vector = Vector3()
    normal_vector.x = self.y * other.z - self.z * other.y
    normal_vector.y = self.z * other.x - self.x * other.z
    normal_vector.z = self.x * other.y - self.y * other.x
    return normal_vector



# 3D 공간에서 TRS(Translation, Rotation, Scale)을 조정하는 4*4 매트릭스
class Matrix4:
  # 1. 생성자 (단위 행렬(identity matrix))
  def __init__(self):
    self.matrix = [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ]

  # 2. 행렬 출력 (소수점 두자리수까지)
  def status(self):
    for row in self.matrix:
      print(f"{row[0]:.2f},{row[1]:.2f},{row[2]:.2f},{row[3]:.2f}")
    print("--"*10)

  # 3. 스케일 조절 (첫 세 행의 각 x, y, z 변형)
  def scale(self,scale_x,scale_y,scale_z):
    self.matrix[0][0] = scale_x
    self.matrix[1][1] = scale_y
    self.matrix[2][2] = scale_z

  # 4, 13. 행렬을 벡터에 적용 ((New) 각 MVP 행렬을 곱해서 모니터 상에 물체가 어느 픽셀에 표현되는지 표시)
  def mul_vector(self, vector):
    # w는 3차원 벡터를 4*4 행렬에서 구현하기 위해 가상으로 만든 개념, 벡터에서 이동값을 담당
    # w가 z의 깊이를 담당하여 원근감 표현을 위해 각 좌표를 나눔
    w = (self.matrix[3][0] * vector.x +
         self.matrix[3][1] * vector.y +
         self.matrix[3][2] * vector.z +
         self.matrix[3][3] * 1.0
         )
    #zero division을 회피 (간단한 클리핑)
    if w == 0:
      w = 0.000001
    new_x = (self.matrix[0][0] * vector.x + self.matrix[0][1] * vector.y + self.matrix[0][2] * vector.z + self.matrix[0][3] * 1.0) / w
    new_y = (self.matrix[1][0] * vector.x + self.matrix[1][1] * vector.y + self.matrix[1][2] * vector.z + self.matrix[1][3] * 1.0) / w
    new_z = (self.matrix[2][0] * vector.x + self.matrix[2][1] * vector.y + self.matrix[2][2] * vector.z + self.matrix[2][3] * 1.0) / w
    return Vector3(new_x, new_y, new_z)

  # 5. 이동 행렬 (mul_vector의 w의 값을 이용하여 위치 벡터를 이동)
  def translate(self, translate_x, translate_y, translate_z):
    # 이러면 기존의 0이었던 4번째 열의 값이 변형되어 x,y,z값이 바뀜
    self.matrix[0][3] = translate_x
    self.matrix[1][3] = translate_y
    self.matrix[2][3] = translate_z

  # 6. z축 회전 행렬 (x, y 행만 변형)
  def rotate_z(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[0][0] = cos
    self.matrix[0][1] = -sin
    self.matrix[1][0] = sin
    self.matrix[1][1] = cos

  #7. y축 회전 행렬 (x, z 행만 변형)
  def rotate_y(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[0][0] = cos
    self.matrix[0][2] = sin
    self.matrix[2][0] = -sin
    self.matrix[2][2] = cos

  #8. x축 회전 행렬 (y, z 행만 변형)
  def rotate_x(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[1][1] = cos
    self.matrix[1][2] = -sin
    self.matrix[2][1] = sin
    self.matrix[2][2] = cos

  #9. 행렬 곱 (수많은 버텍스에 대한 TRS를 동시에 하기 위함(최적화))
  def mul_matrix(self, other):
    result = Matrix4()
    # i = 가로줄(4), j = 세로줄(4)
    for i in range(4):
      for j in range(4):
        # 결과 행렬의 [i][j]는 행렬의 i번째 줄과 다른 행렬의 j번째 줄의 요소끼리의 곱을 더한 값
        result.matrix[i][j] = (
        self.matrix[i][0] * other.matrix[0][j]
        + self.matrix[i][1] * other.matrix[1][j]
        + self.matrix[i][2] * other.matrix[2][j]
        + self.matrix[i][3] * other.matrix[3][j])
    return result

  #10. 오일러 회전 엔진 (약간의 부작용(짐벌락)이 있는 xyz 통합식 회전)
  def euler_rotation(self, x_ang, y_ang, z_ang):
    mat_x = Matrix4()
    mat_x.rotate_x(x_ang)
    mat_y = Matrix4()
    mat_y.rotate_y(y_ang)
    mat_z = Matrix4()
    mat_z.rotate_z(z_ang)
    #X*Y*Z 순서로 회전 행렬들의 곱셈 반환 (회전 순서에 따라 변화가 다름, 엔진마다 순서가 다름)
    mat_yz = mat_z.mul_matrix(mat_y)
    mat_xyz = mat_yz.mul_matrix(mat_x)
    self.matrix = mat_xyz.matrix

  # 11. View 행렬 (카메라의 방향을 확인)
  def view_matrix(self, eye, target, world_yaxis = Vector3(0,1,0)):
    #eye = 월드내 카메라 위치, target = 월드내 목표 위치, world_yaxis = 월드 y축(보통 (0,1,0) 벡터)
    forward = target.minus(eye)            #카메라 정면 방향(z축)
    forward = forward.normalize()
    right = world_yaxis.cross(forward)     #카메라 우측 방향(x축)
    right = right.normalize()
    camera_yaxis = forward.cross(right)    #카메라 상단 방향(y축)
    camera_yaxis = camera_yaxis.normalize()
    #view 행렬 조립
    #카메라는 원점에 고정되어 있기 때문에 세상이 카메라로 와야 함(월드내 카메라 위치를 기준으로 내적 후 카메라 방향으로 다가옴)
    self.matrix = [
        [right.x, right.y, right.z, -right.dot(eye)],
        [camera_yaxis.x, camera_yaxis.y, camera_yaxis.z, -camera_yaxis.dot(eye)],
        [forward.x, forward.y, forward.z, -forward.dot(eye)],
        [0, 0, 0, 1],
    ]

  #12. Projection 행렬
  def project_matrix(self, fov_degree, aspect_ratio, near, far):
    # fov_degree = 시야각, x와 y의 스케일 조정(S), 시야각이 클수록 x, y는 작아짐
    fov_rad = math.radians(fov_degree)
    S = 1.0 / math.tan(fov_rad / 2.0)
    # near, far로 z의 최대 거리, 최소거리 조정
    z_range = far - near
    # 투영 행렬 조립
    self.matrix = [
        [S/aspect_ratio, 0, 0, 0],                # X 크기 (스케일 / 모니터 비율만큼 키움(모니터는 보통 옆으로 길쭉하니까))
        [0, S, 0, 0],                             # Y 크기 (스케일만큼 키움)
        [0,0, far/z_range, -(far*near)/z_range],  # Z-buffer(최소거리와 최대거리 설정)
        [0,0,1,0]                                 # 1이 z의 크기를 복사 후 조정
    ]



Writing mymath.py


In [2]:
# 12일차(26.02.26) (래스터화, 폴리곤 색칠, 파일 분리 및 정리)

%%writefile myrenderer.py

from mymath import *


# 1. ndc_pos = 정규화된 기기 좌표 (각 x, y 축의 화면 끝에서 반대 끝까지를 -1에서 1로 표현(이 수를 벗어나면 화면에서 벗어난 것으로 간주하고 안 보임)
# 기존 매트릭스 클래스에서 분리되어 나왔으므로 독립함수로 유지
def ndc_pos(Project, View, Model, vector3):
  #PVM 순서로 융합 후 위치 벡터를 적용하면 ndc 좌표가 나옴
  return Project.mul_matrix(View).mul_matrix(Model).mul_vector(vector3)

# 2. ndc 값을 받아 실제 모니터 상으로 어느 픽셀에 찍힐지 적용
def pixel_pos(ndc_pos, screen_width = 1920, screen_height = 1080):
  result = Vector3()
  # -1.0 ~ 1.0 사이의 값을 픽셀 비율로 변환
  result.x = int((ndc_pos.x + 1.0) * 0.5 * screen_width)
  # Y축은 그래픽스 수학과 모니터 픽셀(위에서 아래로 내려감) 방향이 반대라서 뒤집어줍니다.
  result.y = int((1.0 - ndc_pos.y) * 0.5 * screen_height)
  result.z = ndc_pos.z
  return result

# 3. 삼각형 외적 (래스터화)(+-로 모니터로 구현된 세개의 점(폴리곤) 안에 해당 픽셀이 포함되는지 확인)(외적한 벡터의 z값이 방향)
def is_in_triangle(self, A, B, C):
  cross1 = (B.minus(A)).cross(self.minus(A)).z    #AB 벡터와 AP(특정 픽셀)벡터의 외적의 z(방향)값
  cross2 = (C.minus(B)).cross(self.minus(B)).z
  cross3 = (A.minus(C)).cross(self.minus(C)).z
  if cross1 >= 0 and cross2 >= 0 and cross3 >= 0:
    return True
  else:
    return False

# 4. 래스터 폴리곤 색칠 (외적의 z값이 면적의 넓이 = 모든 외적을 합치면 그것이 삼각 점의 면적)
def get_pixel_color(self, A, B, C, colorA, colorB, colorC):
  cross1 = (B.minus(A)).cross(self.minus(A)).z   #C의 면적
  cross2 = (C.minus(B)).cross(self.minus(B)).z   #A의 면적
  cross3 = (A.minus(C)).cross(self.minus(C)).z   #B의 면적
  #만약 P가 밖이면 배경색임
  if cross1 < 0 or cross2 < 0 or cross3 < 0:
    return "배경색"
  #전체 면적 = 세 외적의 값
  total_area = cross1 + cross2 + cross3
  #각 외적의 비중
  weight_A = cross1 / total_area
  weight_B = cross2 / total_area
  weight_C = cross3 / total_area
  #각 영향력에 따라 최종 rgb 비율 색상 값 결정
  color_R = (colorA[0]*weight_A) + (colorB[0]*weight_B) + (colorC[0]*weight_C)
  color_G = (colorA[1]*weight_A) + (colorB[1]*weight_B) + (colorC[1]*weight_C)
  color_B = (colorA[2]*weight_A) + (colorB[2]*weight_B) + (colorC[2]*weight_C)
  return f"색상: R: {color_R:.1f}, G: {color_G:.1f}, B: {color_B:.1f}"


#3d 벡터에서 분리되어 버텍스 클래스로 변형
class Vertex:
  def __init__(self, position_vector):
    self.position = position_vector   #위치 벡터
    self.texture = [0.0]              #텍스쳐
    self.color = [255,255,255]        #rgb

Writing myrenderer.py


In [3]:
# 8일차(26.02.20) (4x4 행렬 클래스 구현, 스케일 조절)

from myrenderer import *

#기본 행렬 출력
print("기본 4*4 행렬: ")
new_mat = Matrix4()
new_mat.status()

#스케일 조절된(y축 2배) 벡터 출력
print("y축 2배: ")
new_mat.scale(1.0,2.0,1.0)
new_mat.status()

#행렬에 영향받아 새로운 좌표를 가진 벡터 출력
print("벡터(1,2,3)에 y축 2배의 행렬을 적용: ")
mat_vec = new_mat.mul_vector(Vector3(1.0,2.0,3.0))
mat_vec.status()

# 이동 행렬과 위치 벡터
position_vector = Vector3(1, 3, -5)
print("\n현재 좌표: ")
position_vector.status()
translate_matrix = Matrix4()
translate_matrix.translate(10.0, -2.0, 4.0)
new_position_vector = translate_matrix.mul_vector(position_vector)
print("이동된 좌표: ")
new_position_vector.status()

#z축, y축, x축 회전
z_matrix = Matrix4()
y_matrix = Matrix4()
x_matrix = Matrix4()

z_matrix.rotate_z(90)
y_matrix.rotate_y(90)
x_matrix.rotate_x(90)

r_vector = Vector3(10, 5, 9)
print("\n현재 좌표:")
r_vector.status()

z_vector = z_matrix.mul_vector(r_vector)
y_vector = y_matrix.mul_vector(r_vector)
x_vector = x_matrix.mul_vector(r_vector)

print("\nz축으로 회전된 좌표:")
z_vector.status()
print("y축으로 회전된 좌표:")
y_vector.status()
print("x축으로 회전된 좌표:")
x_vector.status()


기본 4*4 행렬: 
1.00,0.00,0.00,0.00
0.00,1.00,0.00,0.00
0.00,0.00,1.00,0.00
0.00,0.00,0.00,1.00
--------------------
y축 2배: 
1.00,0.00,0.00,0.00
0.00,2.00,0.00,0.00
0.00,0.00,1.00,0.00
0.00,0.00,0.00,1.00
--------------------
벡터(1,2,3)에 y축 2배의 행렬을 적용: 
Vector3(x: 1.00, y: 4.00, z: 3.00)

현재 좌표: 
Vector3(x: 1.00, y: 3.00, z: -5.00)
이동된 좌표: 
Vector3(x: 11.00, y: 1.00, z: -1.00)

현재 좌표:
Vector3(x: 10.00, y: 5.00, z: 9.00)

z축으로 회전된 좌표:
Vector3(x: -5.00, y: 10.00, z: 9.00)
y축으로 회전된 좌표:
Vector3(x: 9.00, y: 5.00, z: -10.00)
x축으로 회전된 좌표:
Vector3(x: 10.00, y: -9.00, z: 5.00)


In [4]:
# 9일차(26.02.23) (행렬 곱, 짐벌락 체험)

from myrenderer import *

# 1. 행렬 곱
print("행렬 곱")
player = Vector3(10,5,0) # 위치 벡터
print("최초 좌표: ")
player.status()

Scale = Matrix4()
Scale.scale(2,2,2) # 스케일 두배
Translate = Matrix4()
Translate.translate(40, 40, 0) # x, y축으로 40씩 이동
Rotate = Matrix4()
Rotate.rotate_z(90) # z축으로 90도 회전

TR_mat = Translate.mul_matrix(Rotate) # 이동 행렬과 회전 행렬의 곱
TRS_mat = TR_mat.mul_matrix(Scale) # TRS를 동시에 실행하는 행렬 (곱셈의 순서와 반대로 SRT순서대로 움직일 거임)

print("\n최종 좌표: ")
final_pos = TRS_mat.mul_vector(player)
final_pos.status() # 최종 좌표
print("="*20)

#2. 짐벌락 체험 (하나의 축이 90도 지점인 순간 다른 축이 똑같이 도는 문제)
print("\n짐벌락 체험: ")
y_axis = Vector3(0,10,0)
print("현재 벡터 방향: ")
y_axis.status()

Mat_A = Matrix4()
Mat_A.euler_rotation(45, 90, 0)
Mat_A.status()
Result_A = Mat_A.mul_vector(y_axis)
print("\n y축 90도, x축 45도 회전된 벡터 방향: ")
Result_A.status()

Mat_B = Matrix4()
Mat_B.euler_rotation(0, 90, -45)
Result_B = Mat_B.mul_vector(y_axis)
print("\n y축 90도, z축 -45도 회전된 벡터 방향: ")
Result_B.status()

행렬 곱
최초 좌표: 
Vector3(x: 10.00, y: 5.00, z: 0.00)

최종 좌표: 
Vector3(x: 30.00, y: 60.00, z: 0.00)

짐벌락 체험: 
현재 벡터 방향: 
Vector3(x: 0.00, y: 10.00, z: 0.00)
0.00,0.71,0.71,0.00
0.00,0.71,-0.71,0.00
-1.00,0.00,0.00,0.00
0.00,0.00,0.00,1.00
--------------------

 y축 90도, x축 45도 회전된 벡터 방향: 
Vector3(x: 7.07, y: 7.07, z: 0.00)

 y축 90도, z축 -45도 회전된 벡터 방향: 
Vector3(x: 7.07, y: 7.07, z: 0.00)


In [5]:
# 10일차 (View 행렬, Projection 행렬)

from myrenderer import *


#1. 카메라 구현(일단 클래스 없이 코드로만)

# 카메라 3요소 세팅
Eye = Vector3(0,5,10)        # 카메라의 현재 위치
Target = Vector3(0,0,0)      # 카메라가 보고 있는 물체의 위치
World_yaxis = Vector3(0,1,0) # 세상의 y축 (카메라의 기울기를 알기 위함)

#카메라의 xyz축(방향)알기
Forward =  Target.minus(Eye)       #카메라의 정면 방향 (보는 물체의 위치 벡터 이후 정규화 - 카메라의 위치 벡터) (카메라의 z축)
Forward = Forward.normalize()
Right = World_yaxis.cross(Forward) #카메라의 오른쪽 방향 (세상의 y축과 정면의 수직(외적) 이후 정규화) (순서 중요, 세상의 y축 -> 정면) (카메라의 x축)
Right = Right.normalize()
Camera_yaxis = Forward.cross(Right) #카메라의 위쪽 방향 (카메라의 정면과 오른쪽 방향의 수직(외적) 이후 정규화) (카메라의 y축)
Camera_yaxis = Camera_yaxis.normalize()

#실험
print("카메라의 x축(우측 방향):")
Right.status()
print("카메라의 y축(상단 방향):")
Camera_yaxis.status()
print("카메라의 z축(정면 방향):")
Forward.status()


#2. View 행렬 테스트

#3개의 요소 구현
Eye = Vector3(0,5,10)
Target = Vector3(0,0,0)
World_yaxis = Vector3(0,1,0)

#행렬로 구현
view_mat = Matrix4()
view_mat.view_matrix(Eye, Target, World_yaxis)
print("="* 20, "\nview 행렬")
view_mat.status()


#3. Projection 행렬 테스트
fov = 60         #시야각
aspect = 16 / 9  #모니터 비율
near = 0.1       #최소 거리
far = 1000       #최대 거리

proj_mat = Matrix4()
proj_mat.project_matrix(fov, aspect, near, far)
print("projection 행렬")
proj_mat.status()

카메라의 x축(우측 방향):
Vector3(x: -1.00, y: 0.00, z: -0.00)
카메라의 y축(상단 방향):
Vector3(x: 0.00, y: 0.89, z: -0.45)
카메라의 z축(정면 방향):
Vector3(x: 0.00, y: -0.45, z: -0.89)
view 행렬
-1.00,0.00,-0.00,-0.00
0.00,0.89,-0.45,-0.00
0.00,-0.45,-0.89,11.18
0.00,0.00,0.00,1.00
--------------------
projection 행렬
0.97,0.00,0.00,0.00
0.00,1.73,0.00,0.00
0.00,0.00,1.00,-0.10
0.00,0.00,1.00,0.00
--------------------


In [6]:
# 11일차(26.02.25) (MVP 융합) (ndc_pos, pixel_pos 함수 적용)

from myrenderer import *

print("MVP 테스트, 몬스터의 눈알은 화면상 어디에 위치해야 할까?")

monster_head = Vector3(0,10,0) #몬스터 키: 10이라 가정
print("월드상 몬스터 눈알의 위치:")
monster_head.status()

# Model: 2배 스케일 조정 및 z축으로 50 이동
mat_scale = Matrix4()
mat_scale.scale(2,2,2)
mat_trans = Matrix4()
mat_trans.translate(0,0,50)
mat_model = Matrix4()
mat_model = mat_trans.mul_matrix(mat_scale)


# View: 카메라 위치는 (0,5,-10), 몬스터 발 위치는 (0,0,50), 월드y축은 (0,1,0)
eye = Vector3(0,5,-10)
target = Vector3(0,0,50)
world_yaxis = Vector3(0,1,0)
mat_view = Matrix4()
mat_view.view_matrix(eye, target, world_yaxis)


# Projection: 시야각은 60, 모니터는 16:9, 최소거리는 0.1, 최대거리는 1000
mat_proj = Matrix4()
mat_proj.project_matrix(60, 16/9, 0.1, 1000)


#ndc 좌표 출력
head_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_head)
print("MVP 적용후 몬스터 눈알의 ndc 좌표:")
head_ndc_pos.status()

#모니터상 찍히는 픽셀
final_head = pixel_pos(head_ndc_pos, 1920,1080)
print("최종적으로 모니터에 찍히는 몬스터 눈알의 좌표")
final_head.status()

MVP 테스트, 몬스터의 눈알은 화면상 어디에 위치해야 할까?
월드상 몬스터 눈알의 위치:
Vector3(x: 0.00, y: 10.00, z: 0.00)
MVP 적용후 몬스터 눈알의 ndc 좌표:
Vector3(x: 0.00, y: 0.59, z: 1.00)
최종적으로 모니터에 찍히는 몬스터 눈알의 좌표
Vector3(x: 960.00, y: 221.00, z: 1.00)


In [7]:
# 12일차(26.02.26) (래스터화, 폴리곤 색상)


from myrenderer import *

#1. 래스터화
#세개의 점 지정 (오른눈, 왼눈, 입)
monster_right_eye = Vector3(3,10,0)
monster_left_eye = Vector3(-3,10,0)
monster_mouth = Vector3(0,8,0)

#세개의 점에 포함되는지 확인할 점
monster_nose = Vector3(0,9,0)
monster_leg = Vector3(4,2,0)

#이전 MVP 그대로 구현
mat_scale = Matrix4()
mat_scale.scale(2,2,2)
mat_trans = Matrix4()
mat_trans.translate(0,0,50)
mat_model = Matrix4()
mat_model = mat_trans.mul_matrix(mat_scale)

eye = Vector3(0,5,-10)
target = Vector3(0,0,50)
world_yaxis = Vector3(0,1,0)
mat_view = Matrix4()
mat_view.view_matrix(eye, target, world_yaxis)

mat_proj = Matrix4()
mat_proj.project_matrix(60, 16/9, 0.1, 1000)

#5개 점 모니터 출력
reye_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_right_eye)
leye_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_left_eye)
mouth_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_mouth)
nose_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_nose)
leg_ndc_pos = ndc_pos(mat_proj, mat_view,mat_model,monster_leg)
final_reye = pixel_pos(reye_ndc_pos)
final_leye = pixel_pos(leye_ndc_pos)
final_mouth = pixel_pos(mouth_ndc_pos)

#각각의 점(코, 다리)이 오른 눈, 왼눈, 입 안에 들어있는지 확인
print(f"몬스터의 코는 오른눈, 왼눈, 입 사이에 있을까?: {is_in_triangle(nose_ndc_pos, reye_ndc_pos,leye_ndc_pos,mouth_ndc_pos)}") # True
print(f"몬스터의 다리는 오른눈, 왼눈, 입 사이에 있을까?: {is_in_triangle(leg_ndc_pos, reye_ndc_pos,leye_ndc_pos,mouth_ndc_pos)}")  #False


#2. 폴리곤 색칠
# 색상 세팅: A는 빨강(255,0,0), B는 초록(0,255,0), C는 파랑(0,0,255)
col_A = [255, 0, 0]
col_B = [0, 255, 0]
col_C = [0, 0, 255]

#입, 오른 눈, 왼 눈의 면적을 차지하는 삼각형의 색상 (코의 위치에 따라 좌지우지됨)
print(get_pixel_color(nose_ndc_pos, reye_ndc_pos, leye_ndc_pos, mouth_ndc_pos, col_A, col_B, col_C))

몬스터의 코는 오른눈, 왼눈, 입 사이에 있을까?: True
몬스터의 다리는 오른눈, 왼눈, 입 사이에 있을까?: False
색상: R: 127.9, G: 63.6, B: 63.6
